In [1]:
!pip install medmnist

In [2]:
!git clone https://github.com/Felix982/XAI-project.git
%cd XAI-project

import os, sys
sys.path.append(os.getcwd())

fatal: destination path 'XAI-project' already exists and is not an empty directory.
/content/XAI-project


In [26]:
!git pull origin main
%cd XAI-project

In [3]:
import os
import sys
import torch

%cd /content/XAI-project
sys.path.append(os.getcwd())

print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

/content/XAI-project
True
Tesla T4


In [4]:
import importlib
import training.train_classifier
importlib.reload(training.train_classifier)
from training.train_classifier import TrainConfig, train_classifier

cfg = TrainConfig(
    data_root="./data",
    image_size=32,
    batch_size=128,
    num_channels=3,
    num_workers=2,
    lr=1e-3,
    max_epochs=30,
    early_stopping_patience=5,
    output_dir="./outputs/classifier",
)

results = train_classifier(cfg)
results

Epoch 001 | train_loss=0.2134 | train_acc=0.9148 | val_loss=1.4047 | val_acc=0.7424
Epoch 002 | train_loss=0.1099 | train_acc=0.9560 | val_loss=0.5067 | val_acc=0.7977
Epoch 003 | train_loss=0.0881 | train_acc=0.9662 | val_loss=0.0893 | val_acc=0.9599
Epoch 004 | train_loss=0.0780 | train_acc=0.9728 | val_loss=0.1605 | val_acc=0.9447
Epoch 005 | train_loss=0.0716 | train_acc=0.9726 | val_loss=0.1107 | val_acc=0.9561
Epoch 006 | train_loss=0.0675 | train_acc=0.9756 | val_loss=0.0798 | val_acc=0.9618
Epoch 007 | train_loss=0.0668 | train_acc=0.9751 | val_loss=0.4060 | val_acc=0.8588
Epoch 008 | train_loss=0.0527 | train_acc=0.9802 | val_loss=0.0663 | val_acc=0.9828
Epoch 009 | train_loss=0.0481 | train_acc=0.9836 | val_loss=0.1031 | val_acc=0.9656
Epoch 010 | train_loss=0.0449 | train_acc=0.9819 | val_loss=0.1092 | val_acc=0.9447
Epoch 011 | train_loss=0.0374 | train_acc=0.9870 | val_loss=0.1795 | val_acc=0.9542
Epoch 012 | train_loss=0.0351 | train_acc=0.9870 | val_loss=0.1928 | val_acc

{'best_epoch': 8,
 'best_val_loss': 0.06634667170241133,
 'test_loss': 0.4532781953995044,
 'test_accuracy': 0.8637820512820513,
 'history': [{'epoch': 1,
   'train_loss': 0.21341035949317905,
   'train_accuracy': 0.9148258283772303,
   'val_loss': 1.4046833578866857,
   'val_accuracy': 0.7423664122137404},
  {'epoch': 2,
   'train_loss': 0.10989981505484933,
   'train_accuracy': 0.9560322854715378,
   'val_loss': 0.5067179312232797,
   'val_accuracy': 0.7977099236641222},
  {'epoch': 3,
   'train_loss': 0.08812239155894938,
   'train_accuracy': 0.9662276975361087,
   'val_loss': 0.08929545142268407,
   'val_accuracy': 0.9599236641221374},
  {'epoch': 4,
   'train_loss': 0.07801996434808486,
   'train_accuracy': 0.9728122344944775,
   'val_loss': 0.16047734408888198,
   'val_accuracy': 0.9446564885496184},
  {'epoch': 5,
   'train_loss': 0.07157353456737434,
   'train_accuracy': 0.9725998300764656,
   'val_loss': 0.11065966216491833,
   'val_accuracy': 0.9561068702290076},
  {'epoch': 

In [6]:
from models.classifier import SmallCNNClassifier

device = "cuda" if torch.cuda.is_available() else "cpu"

model = SmallCNNClassifier(
    in_channels=3,
    base_channels=32,
    dropout=0.1,
).to(device)

ckpt = torch.load("./outputs/classifier/classifier_best.pt", map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

SmallCNNClassifier(
  (features): Sequential(
    (0): ConvBlock(
      (block): Sequential(
        (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU(inplace=True)
        (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (5): ReLU(inplace=True)
        (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
        (7): Dropout(p=0.1, inplace=False)
      )
    )
    (1): ConvBlock(
      (block): Sequential(
        (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU(inplace=True)
        (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (4): BatchNorm2d(

In [7]:
import matplotlib.pyplot as plt
importlib.reload(data.medmnist)
importlib.reload(data.transforms)


test_loader = data.medmnist.get_dataloader(
    split="test",
    batch_size=16,
    image_size=32,
    root="./data",
    shuffle=False,
    num_workers=2,
)

batch = next(iter(test_loader))
images = batch["image"].to(device)
labels = batch["label"].to(device)

with torch.no_grad():
    logits = model(images)
    probs = torch.sigmoid(logits)
    preds = (probs >= 0.5).long()

NameError: name 'data' is not defined

In [ ]:
def denormalize(x):
    return (x * 0.5) + 0.5

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
axes = axes.flatten()

for i in range(16):
    img = denormalize(images[i].cpu()).squeeze().numpy()
    axes[i].imshow(img, cmap="gray")
    axes[i].set_title(
        f"y={labels[i].item()} | p={probs[i].item():.2f} | pred={preds[i].item()}",
        fontsize=8
    )
    axes[i].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
from google.colab import files
files.download("./outputs/classifier/classifier_best.pt")